In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Analitika') \
    .config('spark.jars', '/home/jovyan/jars/hadoop-aws-3.3.4.jar,/home/jovyan/jars/aws-java-sdk-bundle-1.12.262.jar') \
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000') \
    .config('spark.hadoop.fs.s3a.access.key', 'admin') \
    .config('spark.hadoop.fs.s3a.secret.key', 'password123') \
    .config('spark.hadoop.fs.s3a.path.style.access', 'true') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .getOrCreate()

print("Spark MinIO-ya qoşuldu")

Spark MinIO-ya qoşuldu


In [3]:
df_fact = spark.read.parquet("s3a://gold/fact_transaction/")

df_fact.limit(10).toPandas()

,transaction_id,account_sk,branch_sk,customer_sk,amount,currency,channel,status,is_flagged_fraud,transaction_ref,date_key
0,35,68,0,85,2469.43,AZN,Web,Completed,False,TXN_35,20260903
1,38,70,3,9,1821.15,EUR,ATM,Completed,False,TXN_38,20260903
2,65,84,4,61,1659.27,USD,POS,Completed,False,TXN_65,20260903
3,105,4,4,81,1049.31,AZN,ATM,Completed,False,TXN_105,20260903
4,125,48,1,3,1722.00,EUR,POS,Completed,False,TXN_125,20260903
5,126,96,2,19,1394.61,EUR,Mobile,Completed,False,TXN_126,20260903
6,132,68,0,85,534.98,AZN,Web,Completed,False,TXN_132,20260903
7,135,29,2,7,1427.99,AZN,POS,Completed,False,TXN_135,20260903
8,142,0,4,41,1970.18,AZN,POS,Completed,False,TXN_142,20260903
9,201,47,3,15,474.00,AZN,POS,Completed,False,TXN_201,20260903


In [14]:
df_customer = spark.read.parquet("s3a://gold/dim_customer/")

#df_customer.limit(10).toPandas()
df_customer.toPandas()

,customer_id,history_id,address,risk_segment,effective_from,effective_to,is_current,load_run_id,source_system,processed_at,first_name,last_name,email,date_of_birth,prior_credit_rating,current_credit_rating,hash_nk,customer_sk
0,1,101,Old Address 1,Medium,2024-01-01,2025-01-01,False,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_1,Last_1,user_1@example.com,1980-05-30,B,C,hash_customer_1,0
1,1,102,Current Address 1,Medium,2025-01-01,None,True,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_1,Last_1,user_1@example.com,1980-05-30,B,C,hash_customer_1,1
2,2,103,Old Address 2,Low,2024-01-01,2025-01-01,False,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_2,Last_2,user_2@example.com,1980-10-27,B,C,hash_customer_2,2
3,2,104,Current Address 2,Low,2025-01-01,None,True,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_2,Last_2,user_2@example.com,1980-10-27,B,C,hash_customer_2,3
4,3,105,Old Address 3,Medium,2024-01-01,2025-01-01,False,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_3,Last_3,user_3@example.com,1981-03-26,C,A,hash_customer_3,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,48,196,Current Address 48,High,2025-01-01,None,True,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_48,Last_48,user_48@example.com,1999-09-18,A,A,hash_customer_48,95
96,49,197,Old Address 49,Low,2024-01-01,2025-01-01,False,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_49,Last_49,user_49@example.com,2000-02-15,B,C,hash_customer_49,96
97,49,198,Current Address 49,Low,2025-01-01,None,True,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_49,Last_49,user_49@example.com,2000-02-15,B,C,hash_customer_49,97
98,50,199,Old Address 50,Medium,2024-01-01,2025-01-01,False,RUN_2026_09_20_01,POSTGRES_OLTP,2026-09-20 22:46:16,First_50,Last_50,user_50@example.com,2000-07-14,C,A+,hash_customer_50,98


In [12]:
df_fact.createOrReplaceTempView("facts")
df_customer.createOrReplaceTempView("customers")

result = spark.sql("""
    SELECT c.first_name, c.last_name, SUM(f.amount) as total_transaction_amount
    FROM facts f
    JOIN customers c ON f.customer_sk = c.customer_sk
    GROUP BY c.first_name, c.last_name
    ORDER BY total_transaction_amount DESC
""")

result.limit(5).toPandas()

,first_name,last_name,total_transaction_amount
0,First_35,Last_35,111063.90
1,First_4,Last_4,75522.27
2,First_1,Last_1,58621.37
3,First_8,Last_8,54741.95
4,First_13,Last_13,51548.74


In [5]:
df_fact = spark.read.parquet("s3a://gold/fact_transaction/")


df_fact.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- account_sk: long (nullable = true)
 |-- branch_sk: long (nullable = true)
 |-- customer_sk: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- is_flagged_fraud: boolean (nullable = true)
 |-- transaction_ref: string (nullable = true)
 |-- date_key: integer (nullable = true)



In [6]:
df_fact.show(5, truncate=False)

+--------------+----------+---------+-----------+-------+--------+-------+---------+----------------+---------------+--------+
|transaction_id|account_sk|branch_sk|customer_sk|amount |currency|channel|status   |is_flagged_fraud|transaction_ref|date_key|
+--------------+----------+---------+-----------+-------+--------+-------+---------+----------------+---------------+--------+
|35            |68        |0        |85         |2469.43|AZN     |Web    |Completed|false           |TXN_35         |20260903|
|38            |70        |3        |9          |1821.15|EUR     |ATM    |Completed|false           |TXN_38         |20260903|
|65            |84        |4        |61         |1659.27|USD     |POS    |Completed|false           |TXN_65         |20260903|
|105           |4         |4        |81         |1049.31|AZN     |ATM    |Completed|false           |TXN_105        |20260903|
|125           |48        |1        |3          |1722.0 |EUR     |POS    |Completed|false           |TXN_125   

In [7]:
total_rows = df_fact.count()
print(f"Cəmi sətir sayı: {total_rows}")

Cəmi sətir sayı: 1001


In [8]:
df_fact.describe().show()

+-------+------------------+-----------------+------------------+------------------+------------------+--------+-------+---------+---------------+--------------------+
|summary|    transaction_id|       account_sk|         branch_sk|       customer_sk|            amount|currency|channel|   status|transaction_ref|            date_key|
+-------+------------------+-----------------+------------------+------------------+------------------+--------+-------+---------+---------------+--------------------+
|  count|              1001|             1001|              1001|              1001|              1001|    1001|   1001|     1001|           1001|                1001|
|   mean|  508.995004995005|50.54345654345654|1.9150849150849152|46.774225774225776|1260.3282817182815|    NULL|   NULL|     NULL|           NULL|2.0260915172827173E7|
| stddev|289.11680161316286|28.11996372046663|  1.47573107908664|30.855550149445953| 715.5929719850834|    NULL|   NULL|     NULL|           NULL|   8.800176184

In [10]:
from pyspark.sql.functions import col

df_fact.filter(col("amount") > 500).show(5)

+--------------+----------+---------+-----------+-------+--------+-------+---------+----------------+---------------+--------+
|transaction_id|account_sk|branch_sk|customer_sk| amount|currency|channel|   status|is_flagged_fraud|transaction_ref|date_key|
+--------------+----------+---------+-----------+-------+--------+-------+---------+----------------+---------------+--------+
|            35|        68|        0|         85|2469.43|     AZN|    Web|Completed|           false|         TXN_35|20260903|
|            38|        70|        3|          9|1821.15|     EUR|    ATM|Completed|           false|         TXN_38|20260903|
|            65|        84|        4|         61|1659.27|     USD|    POS|Completed|           false|         TXN_65|20260903|
|           105|         4|        4|         81|1049.31|     AZN|    ATM|Completed|           false|        TXN_105|20260903|
|           125|        48|        1|          3| 1722.0|     EUR|    POS|Completed|           false|        TX

In [13]:
df_customer.show()

+-----------+----------+------------------+------------+--------------+------------+----------+-----------------+-------------+-------------------+----------+---------+-------------------+-------------+-------------------+---------------------+----------------+-----------+
|customer_id|history_id|           address|risk_segment|effective_from|effective_to|is_current|      load_run_id|source_system|       processed_at|first_name|last_name|              email|date_of_birth|prior_credit_rating|current_credit_rating|         hash_nk|customer_sk|
+-----------+----------+------------------+------------+--------------+------------+----------+-----------------+-------------+-------------------+----------+---------+-------------------+-------------+-------------------+---------------------+----------------+-----------+
|          1|       101|     Old Address 1|      Medium|    2024-01-01|  2025-01-01|     false|RUN_2026_09_20_01|POSTGRES_OLTP|2026-09-20 22:46:16|   First_1|   Last_1| user_1@ex